In [ ]:
import math

def binomial_tree_american(S0, K, T, r, sigma, N, option_type="call"):
    """
    Binomial Tree for American Option Pricing

    Parameters:
        S0: float        - Initial stock price
        K: float         - Strike price
        T: float         - Time to maturity (in years)
        r: float         - Risk-free interest rate
        sigma: float     - Volatility (standard deviation)
        N: int           - Number of time steps
        option_type: str - "call" or "put"

    Returns:
        float            - Price of the American option
    """
    # Step size
    dt = T / N

    # Up and down factors
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u

    # Risk-neutral probability
    p = (math.exp(r * dt) - d) / (u - d)

    # Initialize asset prices at maturity
    stock_prices = [S0 * (u ** j) * (d ** (N - j)) for j in range(N + 1)]

    # Initialize option values at maturity
    if option_type == "call":
        option_values = [max(price - K, 0) for price in stock_prices]
    elif option_type == "put":
        option_values = [max(K - price, 0) for price in stock_prices]
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # Work backwards through the tree
    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            # Discounted expected value
            expected_value = math.exp(-r * dt) * (p * option_values[j + 1] + (1 - p) * option_values[j])

            # Early exercise value
            stock_price = S0 * (u ** j) * (d ** (i - j))
            if option_type == "call":
                early_exercise = max(stock_price - K, 0)
            else:
                early_exercise = max(K - stock_price, 0)

            # Take the maximum of early exercise and continuation
            option_values[j] = max(expected_value, early_exercise)

    return option_values[0]

# Example usage
S0 = 450   # SPY current price
K = 460    # Strike price
T = 0.5    # Time to maturity in years (6 months)
r = 0.05   # Risk-free rate (5%)
sigma = 0.2  # Volatility (20%)
N = 100    # Number of steps

call_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type="call")
put_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type="put")

print(f"American Call Option Price: {call_price:.2f}")
print(f"American Put Option Price: {put_price:.2f}")


American Call Option Price: 26.03
American Put Option Price: 26.10


In [ ]:
import math

def binomial_tree_american_with_greeks(S0, K, T, r, sigma, N, option_type="call"):
    """
    Binomial Tree for American Option Pricing with Greeks (Delta, Gamma, Theta)

    Parameters:
        S0: float        - Initial stock price
        K: float         - Strike price
        T: float         - Time to maturity (in years)
        r: float         - Risk-free interest rate
        sigma: float     - Volatility (standard deviation)
        N: int           - Number of time steps
        option_type: str - "call" or "put"

    Returns:
        dict: Dictionary containing:
              - "price": Price of the American option
              - "delta": Delta of the option
              - "gamma": Gamma of the option
              - "theta": Theta of the option
    """
    # Step size
    dt = T / N

    # Up and down factors
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u

    # Risk-neutral probability
    p = (math.exp(r * dt) - d) / (u - d)

    # Initialize stock prices at maturity
    stock_prices = [S0 * (u ** j) * (d ** (N - j)) for j in range(N + 1)]

    # Initialize option values at maturity
    if option_type == "call":
        option_values = [max(price - K, 0) for price in stock_prices]
    elif option_type == "put":
        option_values = [max(K - price, 0) for price in stock_prices]
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # Work backwards through the tree
    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            # Discounted expected value
            expected_value = math.exp(-r * dt) * (p * option_values[j + 1] + (1 - p) * option_values[j])

            # Early exercise value
            stock_price = S0 * (u ** j) * (d ** (i - j))
            if option_type == "call":
                early_exercise = max(stock_price - K, 0)
            else:
                early_exercise = max(K - stock_price, 0)

            # Take the maximum of early exercise and continuation
            option_values[j] = max(expected_value, early_exercise)

    # Price of the option
    option_price = option_values[0]

    # Compute Delta
    delta = (option_values[1] - option_values[0]) / (S0 * (u - d))

    # Compute Gamma
    delta_up = (option_values[2] - option_values[1]) / (S0 * u * (u - d))
    delta_down = (option_values[1] - option_values[0]) / (S0 * (d - (1/u)) if (S0 * (d - (1/u))) !=0 else 1)
    gamma = (delta_up - delta_down) / (0.5 * S0 * (u ** 2 - d ** 2))

    # Compute Theta
    theta = (option_values[1] - option_price) / dt

    return {
        "price": option_price,
        "delta": delta,
        "gamma": gamma,
        "theta": theta
    }

# Example usage
S0 = 450   # SPY current price
K = 460    # Strike price
T = 0.5    # Time to maturity in years (6 months)
r = 0.05   # Risk-free rate (5%)
sigma = 0.2  # Volatility (20%)
N = 100    # Number of steps

results_call = binomial_tree_american_with_greeks(S0, K, T, r, sigma, N, option_type="call")
results_put = binomial_tree_american_with_greeks(S0, K, T, r, sigma, N, option_type="put")

print("American Call Option:")
print(f"Price: {results_call['price']:.2f}")
print(f"Delta: {results_call['delta']:.4f}")
print(f"Gamma: {results_call['gamma']:.4f}")
print(f"Theta: {results_call['theta']:.4f}")

print("\nAmerican Put Option:")
print(f"Price: {results_put['price']:.2f}")
print(f"Delta: {results_put['delta']:.4f}")
print(f"Gamma: {results_put['gamma']:.4f}")
print(f"Theta: {results_put['theta']:.4f}")


American Call Option:
Price: 26.03
Delta: 0.2658
Gamma: -0.2433
Theta: 676.5433

American Put Option:
Price: 26.10
Delta: -0.2479
Gamma: 0.2302
Theta: -631.1954


In [ ]:
import numpy as np

def crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type="call"):
    """
    Crank-Nicholson Finite Difference Method for American Options

    Parameters:
        S0: float        - Initial stock price
        K: float         - Strike price
        T: float         - Time to maturity (in years)
        r: float         - Risk-free interest rate
        sigma: float     - Volatility (standard deviation)
        S_max: float     - Maximum stock price for grid boundary
        M: int           - Number of stock price steps
        N: int           - Number of time steps
        option_type: str - "call" or "put"

    Returns:
        float: Price of the American option
    """
    # Grid setup
    dt = T / N                       # Time step size
    dS = S_max / M                   # Stock price step size
    stock_prices = np.linspace(0, S_max, M + 1)  # Stock price grid
    option_values = np.zeros((M + 1))           # Option values at current time
    option_values_new = np.zeros((M + 1))       # Option values at next time step

    # Payoff at maturity
    if option_type == "call":
        option_values[:] = np.maximum(stock_prices - K, 0)
    elif option_type == "put":
        option_values[:] = np.maximum(K - stock_prices, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # Coefficients for Crank-Nicholson scheme
    alpha = 0.25 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 - r * np.arange(M + 1))
    beta = -0.5 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 + r)
    gamma = 0.25 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 + r * np.arange(M + 1))

    # Tridiagonal matrix setup
    A = np.zeros((M - 1, M - 1))
    B = np.zeros((M - 1, M - 1))

    for i in range(M - 1):
        A[i, i] = 1 - beta[i + 1]
        B[i, i] = 1 + beta[i + 1]
        if i > 0:
            A[i, i - 1] = -alpha[i + 1]
            B[i, i - 1] = alpha[i + 1]
        if i < M - 2:
            A[i, i + 1] = -gamma[i + 1]
            B[i, i + 1] = gamma[i + 1]

    # Time-stepping loop
    for n in range(N - 1, -1, -1):
        # Solve B * option_values_new = A * option_values
        rhs = B @ option_values[1:-1]
        option_values_new[1:-1] = np.linalg.solve(A, rhs)

        # Apply early exercise condition
        if option_type == "call":
            option_values_new = np.maximum(option_values_new, stock_prices - K)
        elif option_type == "put":
            option_values_new = np.maximum(option_values_new, K - stock_prices)

        # Update option values
        option_values[:] = option_values_new[:]

    # Interpolation for S0
    i = int(S0 / dS)
    return option_values[i] + (option_values[i + 1] - option_values[i]) * (S0 - stock_prices[i]) / dS

# Example usage
S0 = 450   # SPY current price
K = 460    # Strike price
T = 0.5    # Time to maturity in years (6 months)
r = 0.05   # Risk-free rate (5%)
sigma = 0.2  # Volatility (20%)
S_max = 2 * S0  # Maximum stock price (boundary)
M = 100     # Number of stock price steps
N = 1000    # Number of time steps

call_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type="call")
put_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type="put")

print(f"American Call Option Price: {call_price:.2f}")
print(f"American Put Option Price: {put_price:.2f}")


American Call Option Price: 25.98
American Put Option Price: 26.03


In [ ]:
import numpy as np
def generate_gbm_paths(S0, r, sigma, T, M, N, seed=20):
    """Generate Geometric Brownian Motion paths."""
    np.random.seed(seed)
    dt = T / M
    drift = (r - 0.5 * sigma ** 2) * dt
    diffusion = sigma * np.sqrt(dt) * np.random.normal(size=(M, N))
    log_returns = drift + diffusion
    log_prices = np.cumsum(log_returns, axis=0)
    log_prices = np.vstack((np.zeros(N), log_prices))
    return S0 * np.exp(log_prices)

def least_squares_monte_carlo(S0, K, r, sigma, T, M, N, option_type='call', seed=20):
    """Price American options using Least Squares Monte Carlo."""
    dt = T / M
    discount_factor = np.exp(-r * dt)
    paths = generate_gbm_paths(S0, r, sigma, T, M, N, seed=seed)
    payoffs = np.maximum(paths[-1] - K, 0) if option_type == 'call' else np.maximum(K - paths[-1], 0)

    for t in range(M - 1, 0, -1):
        in_the_money = np.where((paths[t] > K) if option_type == 'call' else (paths[t] < K))[0]
        if len(in_the_money) == 0:
            continue

        X = paths[t, in_the_money]
        Y = payoffs[in_the_money] * discount_factor

        if len(X) > 2:
            X_norm = (X - X.mean()) / X.std()
            try:
                coeffs = np.polyfit(X_norm, Y, deg=2)
                continuation_values = np.polyval(coeffs, (X - X.mean()) / X.std())
            except np.RankWarning:
                continuation_values = np.zeros_like(X)
        else:
            continuation_values = np.zeros_like(X)

        exercise_values = (X - K) if option_type == 'call' else (K - X)
        payoffs[in_the_money] = np.where(exercise_values > continuation_values,
                                        exercise_values,
                                        payoffs[in_the_money]) * discount_factor

    return np.mean(payoffs) * np.exp(-r * T)

In [ ]:
import pandas as pd

# Define the three pricing methods (using your provided functions)

def compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices, option_type="call"):
    """
    Compare American option prices using LSMC, Binomial Tree, and Crank-Nicholson methods.

    Parameters:
        S0: float         - Initial stock price
        K_list: list      - List of strike prices
        T: float          - Time to maturity
        r: float          - Risk-free rate
        sigma: float      - Volatility
        N: int            - Steps for binomial tree
        M: int            - Steps for Crank-Nicholson
        S_max: float      - Maximum stock price for Crank-Nicholson grid
        actual_prices: list - List of actual market prices for comparison
        option_type: str  - "call" or "put"

    Returns:
        DataFrame: A table comparing the methods with actual prices.
    """
    results = []

    # Loop through all strike prices
    for i, K in enumerate(K_list):
        # Least Squares Monte Carlo
        lsmc_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, 10000, option_type=option_type, seed=42)

        # Binomial Tree
        binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type=option_type)

        # Crank-Nicholson
        crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type=option_type)

        # Actual price
        actual_price = actual_prices[i]

        # Append results
        results.append({
            "Strike Price": K,
            "Actual Price": actual_price,
            "LSMC Price": round(lsmc_price, 2),
            "Binomial Tree Price": round(binomial_price, 2),
            "Crank-Nicholson Price": round(crank_price, 2)
        })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Example Usage
if __name__ == "__main__":
    S0 = 450          # Current SPY price
    K_list = options_df['Strike']  # List of strike prices
    T = 0.5           # Time to maturity (6 months)
    r = 0.05          # Risk-free rate
    sigma = 0.2       # Volatility
    N = 100           # Steps for Binomial Tree
    M = 100           # Steps for Crank-Nicholson
    S_max = 2 * S0    # Maximum stock price for Crank-Nicholson

    # Actual prices for comparison (example values, replace with real market data)
    actual_prices = options_df['Strike']
    # Run the comparison
    comparison_table = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices, option_type="call")

    # Display the results
    print(comparison_table)

    Strike Price  Actual Price  LSMC Price  Binomial Tree Price  \
0            588           588        1.16                 1.26   
1            589           589        1.15                 1.22   
2            590           590        1.10                 1.19   
3            591           591        1.08                 1.15   
4            592           592        1.05                 1.12   
5            593           593        1.02                 1.08   
6            594           594        0.99                 1.05   
7            595           595        0.96                 1.01   
8            596           596        0.93                 0.98   
9            597           597        0.89                 0.94   
10           598           598        0.88                 0.92   
11           599           599        0.84                 0.90   
12           600           600        0.85                 0.87   
13           601           601        0.82                 0.8

In [ ]:
import pandas as pd

# Define the three pricing methods (using your provided functions)
def compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices, option_type="call"):
    """
    Compare American option prices using LSMC, Binomial Tree, and Crank-Nicholson methods.

    Parameters:
        S0: float         - Initial stock price
        K_list: list      - List of strike prices
        T: float          - Time to maturity
        r: float          - Risk-free rate
        sigma: float      - Volatility
        N: int            - Steps for binomial tree
        M: int            - Steps for Crank-Nicholson
        S_max: float      - Maximum stock price for Crank-Nicholson grid
        actual_prices: list - List of actual market prices for comparison
        option_type: str  - "call" or "put"

    Returns:
        DataFrame: A table comparing the methods with actual prices.
    """
    results = []

    # Loop through all strike prices
    for i, K in enumerate(K_list):
        # Least Squares Monte Carlo
        lsmc_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, 10000, option_type=option_type, seed=42)

        # Binomial Tree
        binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type=option_type)

        # Crank-Nicholson
        crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type=option_type)

        # Actual price
        actual_price = actual_prices[i]

        # Append results
        results.append({
            "Strike Price": K,
            "Actual Price": actual_price,
            "LSMC Price": round(lsmc_price, 2),
            "Binomial Tree Price": round(binomial_price, 2),
            "Crank-Nicholson Price": round(crank_price, 2)
        })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Example Usage
if __name__ == "__main__":
    # Example of how to extract the required data from options_df
    #options_df = pd.DataFrame({
        #'Strike': [588, 589, 590, 591, 592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610],
        #'Actual_Price_Call': [12.18, 11.49, 10.86, 10.21, 9.58],  # Example Call prices
        #'Actual_Price_Put': [7.75, 8.11, 8.48, 8.86, 9.27]  # Example Put prices
    #})

    S0 = 590.50  # Current SPY price (Spot price)
    K_list = options_df['Strike']  # List of strike prices
    T = 30/360  # Time to maturity (6 months)
    r = 0.0423  # Risk-free rate
    sigma = 0.14  # Volatility
    N = 100  # Steps for Binomial Tree
    M = 100  # Steps for Crank-Nicholson
    S_max = 2 * S0  # Maximum stock price for Crank-Nicholson

    # Actual prices for comparison (from the DataFrame)
    actual_prices_call = options_df['Last']
    actual_prices_put = options_df['Actual_Price_Put']

    # Run the comparison for Call options
    comparison_table_call = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_call, option_type="call")
    print("Call Option Pricing Comparison:")
    print(comparison_table_call)

    # Run the comparison for Put options
    comparison_table_put = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_put, option_type="put")
    print("\nPut Option Pricing Comparison:")
    print(comparison_table_put)


KeyError: 'Last'

In [ ]:
import pandas as pd

# Define the three pricing methods (using your provided functions)
def compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices, option_type="call"):
    """
    Compare American option prices using LSMC, Binomial Tree, and Crank-Nicholson methods.

    Parameters:
        S0: float         - Initial stock price
        K_list: list      - List of strike prices
        T: float          - Time to maturity
        r: float          - Risk-free rate
        sigma: float      - Volatility
        N: int            - Steps for binomial tree
        M: int            - Steps for Crank-Nicholson
        S_max: float      - Maximum stock price for Crank-Nicholson grid
        actual_prices: list - List of actual market prices for comparison
        option_type: str  - "call" or "put"

    Returns:
        DataFrame: A table comparing the methods with actual prices.
    """
    results = []

    # Loop through all strike prices
    for i, K in enumerate(K_list):
        # Least Squares Monte Carlo
        lsmc_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, 10000, option_type=option_type, seed=42)

        # Binomial Tree
        binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type=option_type)

        # Crank-Nicholson
        crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type=option_type)

        # Actual price
        actual_price = actual_prices[i]

        # Append results
        results.append({
            "Strike Price": K,
            "Actual Price": actual_price,
            "LSMC Price": round(lsmc_price, 2),
            "Binomial Tree Price": round(binomial_price, 2),
            "Crank-Nicholson Price": round(crank_price, 2)
        })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Example Usage
if __name__ == "__main__":
    # Data you provided in DataFrame format (with strike prices and actual call/put prices)
    options_data = {
        'Strike': [588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612],
        'Call Price': [12.18, 11.49, 10.86, 10.21, 9.58, 8.98, 8.40, 7.86, 7.32, 6.80, 6.31, 5.86, 5.41, 4.99, 4.60, 4.21, 3.87, 3.55, 3.23, 2.94, 2.68, 2.43, 2.21, 2.00, 1.81],
        'Put Price': [7.75, 8.11, 8.48, 8.86, 9.27, 9.68, 10.14, 10.61, 11.11, 11.60, 11.68, 12.30, 13.02, 13.30, 13.92, 14.57, 15.24, 15.94, 16.67, 17.40, 18.15, 18.94, 19.73, 20.55, 21.39]
    }

    options_df = pd.DataFrame(options_data)

    S0 = 590.50  # Current SPY price (Spot price)
    K_list = options_df['Strike']  # List of strike prices
    T = 30/360  # Time to maturity (6 months)
    r = 0.0423  # Risk-free rate
    sigma = 0.14  # Volatility
    N = 100  # Steps for Binomial Tree
    M = 100  # Steps for Crank-Nicholson
    S_max = 2 * S0  # Maximum stock price for Crank-Nicholson

    # Run the comparison for Call options
    actual_prices_call = options_df['Call Price']
    comparison_table_call = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_call, option_type="call")
    print("Call Option Pricing Comparison:")
    print(comparison_table_call)

    # Run the comparison for Put options
    actual_prices_put = options_df['Put Price']
    comparison_table_put = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_put, option_type="put")
    print("\nPut Option Pricing Comparison:")
    print(comparison_table_put)


Call Option Pricing Comparison:
    Strike Price  Actual Price  LSMC Price  Binomial Tree Price  \
0            588         12.18       11.64                11.97   
1            589         11.49       11.08                11.40   
2            590         10.86       10.57                10.84   
3            591         10.21       10.02                10.31   
4            592          9.58        9.52                 9.83   
5            593          8.98        9.07                 9.34   
6            594          8.40        8.61                 8.86   
7            595          7.86        8.12                 8.37   
8            596          7.32        7.71                 7.94   
9            597          6.80        7.28                 7.53   
10           598          6.31        6.88                 7.13   
11           599          5.86        6.51                 6.72   
12           600          5.41        6.14                 6.31   
13           601          4.99

In [ ]:
import pandas as pd

# The provided text data
data = """
588 SPY 12/20/24 C588 12.18000031 12.21000004 12.15999985 14.19703293 1472 588 SPY 12/20/24 P588 7.75 7.789999962 7.780000687 14.44931126 1956
589 SPY 12/20/24 C589 11.48999977 11.53999996 11.56999969 14.01831722 1869 589 SPY 12/20/24 P589 8.109999657 8.140000343 8.119999886 14.30964947 1291
590 SPY 12/20/24 C590 10.85999966 10.88000011 10.85000038 13.86997414 5497 590 SPY 12/20/24 P590 8.479999542 8.5 8.510000229 14.16133308 2473
591 SPY 12/20/24 C591 10.21000004 10.23999977 10.14000034 13.698246 2308 591 SPY 12/20/24 P591 8.859999657 8.899999619 8.829999924 14.02643871 698
592 SPY 12/20/24 C592 9.579999924 9.630000114 9.550000191 13.54008293 1265 592 SPY 12/20/24 P592 9.270000458 9.31000042 9.220000267 13.89594078 1275
593 SPY 12/20/24 C593 8.979999542 9.029999733 8.829999924 13.38659286 1219 593 SPY 12/20/24 P593 9.680000305 9.729999542 9.779999733 13.74718666 740
594 SPY 12/20/24 C594 8.399999619 8.449999809 8.199999809 13.23713493 568 594 SPY 12/20/24 P594 10.14000034 10.18000031 10.60999966 13.63123226 1242
595 SPY 12/20/24 C595 7.86000061 7.88999939 7.840000153 13.1059494 2479 595 SPY 12/20/24 P595 10.60999966 10.64999962 10.61999989 13.51008797 1470
596	SPY 12/20/24 C596	7.320000648	7.369999886	7.320000648	12.97714138	610	596	SPY 12/20/24 P596	11.10999966	11.14999962	11.43000031	13.40071869	228
597	SPY 12/20/24 C597	6.800000191	6.850000381	6.820000648	12.8357439	611	597	SPY 12/20/24 P597	11.60000038	11.65999985	12.14999962	13.26738453	124
598	SPY 12/20/24 C598	6.31000042	6.36000061	6.199999809	12.71089077	879	598	SPY 12/20/24 P598	11.68000031	12.55000019	14.47000027	13.07985497	10
599	SPY 12/20/24 C599	5.86000061	5.890000343	5.760000229	12.60215855	1451	599	SPY 12/20/24 P599	12.30000019	13.10999966	13.40999985	13.0215807	31
600	SPY 12/20/24 C600	5.409999847	5.440000534	5.390000343	12.47955036	7804	600	SPY 12/20/24 P600	13.02000046	13.68999958	13.19999981	13.02745533	192
601	SPY 12/20/24 C601	4.989999771	5.030000687	5.010000229	12.38053131	586	601	SPY 12/20/24 P601	13.30000019	14.40999985	15.60000038	12.76427937	15
602	SPY 12/20/24 C602	4.600000381	4.630000114	4.570000648	12.28307724	891	602	SPY 12/20/24 P602	13.92000008	15.10000038	0	12.71119881	0
603	SPY 12/20/24 C603	4.210000038	4.260000229	4.280000687	12.17814159	1883	603	SPY 12/20/24 P603	14.56999969	15.77000046	0	12.6351099	0
604	SPY 12/20/24 C604	3.869999886	3.900000572	3.890000343	12.09124088	1001	604	SPY 12/20/24 P604	15.23999977	16.36999512	16.6000061	12.47722912	2
605	SPY 12/20/24 C605	3.550000191	3.569999695	3.529999733	12.01469898	3717	605	SPY 12/20/24 P605	15.93999958	16.81999207	16.80000305	12.16010284	63
606	SPY 12/20/24 C606	3.230000496	3.270000458	3.220000267	11.93249035	1117	606	SPY 12/20/24 P606	16.66999817	17.84999084	0	12.38138294	0
607	SPY 12/20/24 C607	2.940000534	2.980000496	2.940000534	11.85333347	1172	607	SPY 12/20/24 P607	17.3999939	18.6000061	0	12.29963112	0
608	SPY 12/20/24 C608	2.680000305	2.720000267	2.680000305	11.79769325	502	608	SPY 12/20/24 P608	18.1499939	19.36000061	0	12.20449734	0
609	SPY 12/20/24 C609	2.430000305	2.470000267	2.5	11.72910213	984	609	SPY 12/20/24 P609	18.94000244	20.11999512	22.43000793	12.10474586	2
610	SPY 12/20/24 C610	2.210000038	2.239999771	2.260000229	11.67756367	4480	610	SPY 12/20/24 P610	19.72999573	20.80000305	24.19999695	11.84862423	201
"""

# Define columns
columns = [
    'Strike', 'Option_Type', 'Last', 'Bid', 'Ask', 'Implied_Vol', 'Volume'
]

call_data, put_data = [], []

# Split and parse rows
for row in data.strip().split('\n'):
    items = row.split()

    # Extract call and put data
    call = [items[0], 'Call', items[4], items[5], items[6], items[7], items[8]]
    put = [items[0], 'Put', items[12], items[13], items[14], items[15], items[16]]

    call_data.append(call)
    put_data.append(put)

# Combine into DataFrames
call_df = pd.DataFrame(call_data, columns=columns)
put_df = pd.DataFrame(put_data, columns=columns)

# Combine both call and put data
options_df = pd.concat([call_df, put_df], ignore_index=True)

# Display the final DataFrame
print(options_df)


   Strike Option_Type         Last          Bid          Ask  Implied_Vol  \
0     588        Call  12.18000031  12.21000004  12.15999985  14.19703293   
1     589        Call  11.48999977  11.53999996  11.56999969  14.01831722   
2     590        Call  10.85999966  10.88000011  10.85000038  13.86997414   
3     591        Call  10.21000004  10.23999977  10.14000034    13.698246   
4     592        Call  9.579999924  9.630000114  9.550000191  13.54008293   
5     593        Call  8.979999542  9.029999733  8.829999924  13.38659286   
6     594        Call  8.399999619  8.449999809  8.199999809  13.23713493   
7     595        Call   7.86000061   7.88999939  7.840000153   13.1059494   
8     596        Call  7.320000648  7.369999886  7.320000648  12.97714138   
9     597        Call  6.800000191  6.850000381  6.820000648   12.8357439   
10    598        Call   6.31000042   6.36000061  6.199999809  12.71089077   
11    599        Call   5.86000061  5.890000343  5.760000229  12.60215855   

In [ ]:
import pandas as pd

# The provided text data
data = """
SPY 3/21/25 C575 575 42.220001 42.389999 42.750000 17.130119 19 SPY 3/21/25 P575 575 7.140000 7.190001 7.140000 16.426985 1189
SPY 3/21/25 C580 580 38.040009 38.229996 38.589996 16.517515 15 SPY 3/21/25 P580 580 7.960000 8.000000 7.970000 15.857189 513
SPY 3/21/25 C585 585 33.970001 34.150009 34.180008 15.898024 20 SPY 3/21/25 P585 585 8.890000 8.930000 8.890000 15.285378 665
SPY 3/21/25 C590 590 30.019989 30.199997 30.139999 15.286833 101 SPY 3/21/25 P590 590 9.930000 9.990000 9.850000 14.697979 786
SPY 3/21/25 C595 595 26.210007 26.399994 26.630005 14.686543 39 SPY 3/21/25 P595 595 11.170000 11.210000 11.100000 14.127433 1631
SPY 3/21/25 C600 600 22.580002 22.759995 22.699997 14.102382 1910 SPY 3/21/25 P600 600 12.590000 12.630000 12.630000 13.551406 2695
SPY 3/21/25 C605 605 19.130005 19.330002 19.410004 13.529965 833 SPY 3/21/25 P605 605 14.270000 14.300000 14.250000 13.001152 449
SPY 3/21/25 C610 610 16.000000 16.169998 16.130005 13.027735 527 SPY 3/21/25 P610 610 16.240005 16.290009 15.940000 12.474047 267
SPY 3/21/25 C615 615 13.220000 13.279999 13.250000 12.584682 198 SPY 3/21/25 P615 615 18.469986 18.799988 18.149994 12.007297 18
SPY 3/21/25 C620 620 10.680000 10.730000 10.770000 12.145025 1274 SPY 3/21/25 P620 620 21.250000 21.889999 22.009995 11.721725 41
"""

# Define the columns
columns = [
    'Call Ticker', 'Call Date', 'Call Option_Type', 'Call Strike', 'Call Last', 'Call Bid', 'Call Ask', 'Call Implied_Vol', 'Call Volume',
    'Put Ticker', 'Put Date', 'Put Option_Type', 'Put Strike', 'Put Last', 'Put Bid', 'Put Ask', 'Put Implied_Vol', 'Put Volume'
]

# Split and parse rows
data_rows = []
for row in data.strip().split('\n'):
    items = row.split()

    # Extract data for Call and Put options
    call_data = [items[0], items[1], 'Call', items[2], items[3], items[4], items[5], items[6], items[7]]
    put_data = [items[0], items[1], 'Put', items[2], items[8], items[9], items[10], items[11], items[12]]

    # Combine both Call and Put data into a single row
    data_rows.append(call_data + put_data)

# Create a DataFrame
options_df2 = pd.DataFrame(data_rows, columns=columns)

# Display the final DataFrame
print(options_df2)


  Call Ticker Call Date Call Option_Type Call Strike Call Last   Call Bid  \
0         SPY   3/21/25             Call        C575       575  42.220001   
1         SPY   3/21/25             Call        C580       580  38.040009   
2         SPY   3/21/25             Call        C585       585  33.970001   
3         SPY   3/21/25             Call        C590       590  30.019989   
4         SPY   3/21/25             Call        C595       595  26.210007   
5         SPY   3/21/25             Call        C600       600  22.580002   
6         SPY   3/21/25             Call        C605       605  19.130005   
7         SPY   3/21/25             Call        C610       610  16.000000   
8         SPY   3/21/25             Call        C615       615  13.220000   
9         SPY   3/21/25             Call        C620       620  10.680000   

    Call Ask Call Implied_Vol Call Volume Put Ticker Put Date Put Option_Type  \
0  42.389999        42.750000   17.130119        SPY  3/21/25          

In [ ]:
import pandas as pd

# Define the three pricing methods (using your provided functions)
def compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices, option_type="call"):
    """
    Compare American option prices using LSMC, Binomial Tree, and Crank-Nicholson methods.

    Parameters:
        S0: float         - Initial stock price
        K_list: list      - List of strike prices
        T: float          - Time to maturity
        r: float          - Risk-free rate
        sigma: float      - Volatility
        N: int            - Steps for binomial tree
        M: int            - Steps for Crank-Nicholson
        S_max: float      - Maximum stock price for Crank-Nicholson grid
        actual_prices: list - List of actual market prices for comparison
        option_type: str  - "call" or "put"

    Returns:
        DataFrame: A table comparing the methods with actual prices.
    """
    results = []

    # Loop through all strike prices
    for i, K in enumerate(K_list):
        # Least Squares Monte Carlo
        lsmc_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, 10000, option_type=option_type, seed=42)

        # Binomial Tree
        binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type=option_type)

        # Crank-Nicholson
        crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type=option_type)

        # Actual price
        actual_price = actual_prices[i]

        # Append results
        results.append({
            "Strike Price": K,
            "Actual Price": actual_price,
            "LSMC Price": round(lsmc_price, 2),
            "Binomial Tree Price": round(binomial_price, 2),
            "Crank-Nicholson Price": round(crank_price, 2)
        })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Example Usage
if __name__ == "__main__":
    # Data you provided in DataFrame format (with strike prices and actual call/put prices)


# Define the options data as a dictionary
    options_data = {
    'Strike': [575, 580, 585, 590, 595, 600, 605, 610, 615, 620],
    'Call Price': [42.22, 38.04, 33.97, 30.02, 26.21, 22.58, 19.13, 16.00, 13.22, 10.68],
    'Put Price': [7.14, 7.96, 8.89, 9.93, 11.17, 12.59, 14.27, 16.24, 18.47, 21.25]
}




    options_df = pd.DataFrame(options_data)
#13 DEC
    S0 = 604.21  # Current SPY price (Spot price)
    K_list = options_df['Strike']  # List of strike prices
    T = 98/360  # Time to maturity (6 months)
    r = 0.0423  # Risk-free rate
    sigma = 0.14  # Volatility
    N = 100  # Steps for Binomial Tree
    M = 100  # Steps for Crank-Nicholson
    S_max = 2 * S0  # Maximum stock price for Crank-Nicholson

    # Run the comparison for Call options
    actual_prices_call = options_df['Call Price']
    comparison_table_call = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_call, option_type="call")
    print("Call Option Pricing Comparison:")
    print(comparison_table_call)

    # Run the comparison for Put options
    actual_prices_put = options_df['Put Price']
    comparison_table_put = compare_option_pricing(S0, K_list, T, r, sigma, N, M, S_max, actual_prices_put, option_type="put")
    print("\nPut Option Pricing Comparison:")
    print(comparison_table_put)


Call Option Pricing Comparison:
   Strike Price  Actual Price  LSMC Price  Binomial Tree Price  \
0           575         42.22       39.27                40.64   
1           580         38.04       35.50                36.82   
2           585         33.97       31.92                33.20   
3           590         30.02       28.67                29.80   
4           595         26.21       25.61                26.52   
5           600         22.58       22.63                23.58   
6           605         19.13       19.90                20.74   
7           610         16.00       17.42                18.22   
8           615         13.22       15.19                15.86   
9           620         10.68       13.13                13.74   

   Crank-Nicholson Price  
0                  40.63  
1                  36.69  
2                  33.19  
3                  29.70  
4                  26.50  
5                  23.52  
6                  20.62  
7                  18.19 

In [ ]:
# Binomial Tree Model for American Option Pricing
def binomial_tree_american(S0, K, T, r, sigma, N, option_type="call"):
    dt = T / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp(r * dt) - d) / (u - d)

    stock_prices = [S0 * (u ** j) * (d ** (N - j)) for j in range(N + 1)]
    if option_type == "call":
        option_values = [max(price - K, 0) for price in stock_prices]
    elif option_type == "put":
        option_values = [max(K - price, 0) for price in stock_prices]
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            expected_value = math.exp(-r * dt) * (p * option_values[j + 1] + (1 - p) * option_values[j])
            stock_price = S0 * (u ** j) * (d ** (i - j))
            early_exercise = max(stock_price - K, 0) if option_type == "call" else max(K - stock_price, 0)
            option_values[j] = max(expected_value, early_exercise)

    return option_values[0]

# Crank-Nicholson Finite Difference Model for American Option Pricing
def crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type="call"):
    dt = T / N
    dS = S_max / M
    stock_prices = np.linspace(0, S_max, M + 1)
    option_values = np.zeros((M + 1))
    option_values_new = np.zeros((M + 1))

    if option_type == "call":
        option_values[:] = np.maximum(stock_prices - K, 0)
    elif option_type == "put":
        option_values[:] = np.maximum(K - stock_prices, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    alpha = 0.25 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 - r * np.arange(M + 1))
    beta = -0.5 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 + r)
    gamma = 0.25 * dt * (sigma ** 2 * np.arange(M + 1) ** 2 + r * np.arange(M + 1))

    A = np.zeros((M - 1, M - 1))
    B = np.zeros((M - 1, M - 1))

    for i in range(M - 1):
        A[i, i] = 1 - beta[i + 1]
        B[i, i] = 1 + beta[i + 1]
        if i > 0:
            A[i, i - 1] = -alpha[i + 1]
            B[i, i - 1] = alpha[i + 1]
        if i < M - 2:
            A[i, i + 1] = -gamma[i + 1]
            B[i, i + 1] = gamma[i + 1]

    for n in range(N - 1, -1, -1):
        rhs = B @ option_values[1:-1]
        option_values_new[1:-1] = np.linalg.solve(A, rhs)
        if option_type == "call":
            option_values_new = np.maximum(option_values_new, stock_prices - K)
        elif option_type == "put":
            option_values_new = np.maximum(option_values_new, K - stock_prices)

        option_values[:] = option_values_new[:]

    i = int(S0 / dS)
    return option_values[i] + (option_values[i + 1] - option_values[i]) * (S0 - stock_prices[i]) / dS

# Least Squares Monte Carlo for American Option Pricing
def generate_gbm_paths(S0, r, sigma, T, M, N, seed=20):
    np.random.seed(seed)
    dt = T / M
    drift = (r - 0.5 * sigma ** 2) * dt
    diffusion = sigma * np.sqrt(dt) * np.random.normal(size=(M, N))
    log_returns = drift + diffusion
    log_prices = np.cumsum(log_returns, axis=0)
    log_prices = np.vstack((np.zeros(N), log_prices))
    return S0 * np.exp(log_prices)

def least_squares_monte_carlo(S0, K, r, sigma, T, M, N, option_type='call', seed=20):
    dt = T / M
    discount_factor = np.exp(-r * dt)
    paths = generate_gbm_paths(S0, r, sigma, T, M, N, seed=seed)
    payoffs = np.maximum(paths[-1] - K, 0) if option_type == 'call' else np.maximum(K - paths[-1], 0)

    for t in range(M - 1, 0, -1):
        in_the_money = np.where((paths[t] > K) if option_type == 'call' else (paths[t] < K))[0]
        if len(in_the_money) == 0:
            continue

        X = paths[t, in_the_money]
        Y = payoffs[in_the_money] * discount_factor

        if len(X) > 2:
            X_norm = (X - X.mean()) / X.std()
            try:
                coeffs = np.polyfit(X_norm, Y, deg=2)
                continuation_values = np.polyval(coeffs, (X - X.mean()) / X.std())
            except np.RankWarning:
                continuation_values = np.zeros_like(X)
        else:
            continuation_values = np.zeros_like(X)

        exercise_values = (X - K) if option_type == 'call' else (K - X)
        payoffs[in_the_money] = np.where(exercise_values > continuation_values,
                                        exercise_values,
                                        payoffs[in_the_money]) * discount_factor

    return np.mean(payoffs) * np.exp(-r * T)

Computatation of greeks


In [ ]:
import math
import numpy as np



# Greeks Calculations for Binomial Tree
def binomial_tree_delta(S0, K, T, r, sigma, N, option_type="call", delta_S=0.01*S0):
    option_price_up = binomial_tree_american(S0 + delta_S, K, T, r, sigma, N, option_type)
    option_price_down = binomial_tree_american(S0 - delta_S, K, T, r, sigma, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_S)

def binomial_tree_gamma(S0, K, T, r, sigma, N, option_type="call", delta_S=0.01*S0):
    option_price_up = binomial_tree_american(S0 + delta_S, K, T, r, sigma, N, option_type)
    option_price_down = binomial_tree_american(S0 - delta_S, K, T, r, sigma, N, option_type)
    option_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type)
    return (option_price_up - 2 * option_price + option_price_down) / (delta_S ** 2)

def binomial_tree_vega(S0, K, T, r, sigma, N, option_type="call", delta_sigma=0.01*S0):
    option_price_up = binomial_tree_american(S0, K, T, r, sigma + delta_sigma, N, option_type)
    option_price_down = binomial_tree_american(S0, K, T, r, sigma - delta_sigma, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_sigma)

def binomial_tree_theta(S0, K, T, r, sigma, N, option_type="call", delta_T=0.01):
    option_price_up = binomial_tree_american(S0, K, T - delta_T, r, sigma, N, option_type)
    option_price_down = binomial_tree_american(S0, K, T + delta_T, r, sigma, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_T)

def binomial_tree_rho(S0, K, T, r, sigma, N, option_type="call", delta_r=0.01*S0):
    option_price_up = binomial_tree_american(S0, K, T, r + delta_r, sigma, N, option_type)
    option_price_down = binomial_tree_american(S0, K, T, r - delta_r, sigma, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_r)

# Greeks Calculations for Crank-Nicholson Finite Difference
def crank_nicholson_delta(S0, K, T, r, sigma, S_max, M, N, option_type="call", delta_S=0.01*S0):
    option_price_up = crank_nicholson_american(S0 + delta_S, K, T, r, sigma, S_max, M, N, option_type)
    option_price_down = crank_nicholson_american(S0 - delta_S, K, T, r, sigma, S_max, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_S)

def crank_nicholson_gamma(S0, K, T, r, sigma, S_max, M, N, option_type="call", delta_S=0.01*S0):
    option_price_up = crank_nicholson_american(S0 + delta_S, K, T, r, sigma, S_max, M, N, option_type)
    option_price_down = crank_nicholson_american(S0 - delta_S, K, T, r, sigma, S_max, M, N, option_type)
    option_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type)
    return (option_price_up - 2 * option_price + option_price_down) / (delta_S ** 2)

def crank_nicholson_vega(S0, K, T, r, sigma, S_max, M, N, option_type="call", delta_sigma=0.01*S0):
    option_price_up = crank_nicholson_american(S0, K, T, r, sigma + delta_sigma, S_max, M, N, option_type)
    option_price_down = crank_nicholson_american(S0, K, T, r, sigma - delta_sigma, S_max, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_sigma)

def crank_nicholson_theta(S0, K, T, r, sigma, S_max, M, N, option_type="call", delta_T=0.01*S0):
    option_price_up = crank_nicholson_american(S0, K, T - delta_T, r, sigma, S_max, M, N, option_type)
    option_price_down = crank_nicholson_american(S0, K, T + delta_T, r, sigma, S_max, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_T)

def crank_nicholson_rho(S0, K, T, r, sigma, S_max, M, N, option_type="call", delta_r=0.01*S0):
    option_price_up = crank_nicholson_american(S0, K, T, r + delta_r, sigma, S_max, M, N, option_type)
    option_price_down = crank_nicholson_american(S0, K, T, r - delta_r, sigma, S_max, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_r)

# Greeks Calculations for Least Squares Monte Carlo
def least_squares_monte_carlo_delta(S0, K, r, sigma, T, M, N, option_type='call', delta_S=0.01*S0):
    option_price_up = least_squares_monte_carlo(S0 + delta_S, K, r, sigma, T, M, N, option_type)
    option_price_down = least_squares_monte_carlo(S0 - delta_S, K, r, sigma, T, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_S)

def least_squares_monte_carlo_gamma(S0, K, r, sigma, T, M, N, option_type='call', delta_S=0.01*S0):
    option_price_up = least_squares_monte_carlo(S0 + delta_S, K, r, sigma, T, M, N, option_type)
    option_price_down = least_squares_monte_carlo(S0 - delta_S, K, r, sigma, T, M, N, option_type)
    option_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, N, option_type)
    return (option_price_up - 2 * option_price + option_price_down) / (delta_S ** 2)

def least_squares_monte_carlo_vega(S0, K, r, sigma, T, M, N, option_type='call', delta_sigma=0.01*S0):
    option_price_up = least_squares_monte_carlo(S0, K, r, sigma + delta_sigma, T, M, N, option_type)
    option_price_down = least_squares_monte_carlo(S0, K, r, sigma - delta_sigma, T, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_sigma)

def least_squares_monte_carlo_theta(S0, K, r, sigma, T, M, N, option_type='call', delta_T=0.01*S0):
    option_price_up = least_squares_monte_carlo(S0, K, r, sigma, T - delta_T, M, N, option_type)
    option_price_down = least_squares_monte_carlo(S0, K, r, sigma, T + delta_T, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_T)

def least_squares_monte_carlo_rho(S0, K, r, sigma, T, M, N, option_type='call', delta_r=0.01*S0):
    option_price_up = least_squares_monte_carlo(S0, K, r + delta_r, sigma, T, M, N, option_type)
    option_price_down = least_squares_monte_carlo(S0, K, r - delta_r, sigma, T, M, N, option_type)
    return (option_price_up - option_price_down) / (2 * delta_r)


In [ ]:
import math
import numpy as np

# Function to run all models and compare their results
def compare_option_pricing(S0, K, T, r, sigma, N, M, S_max, option_type="call"):
    # Binomial Tree Model
    binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type)
    binomial_delta = binomial_tree_delta(S0, K, T, r, sigma, N, option_type)
    binomial_gamma = binomial_tree_gamma(S0, K, T, r, sigma, N, option_type)
    binomial_vega = binomial_tree_vega(S0, K, T, r, sigma, N, option_type)
    binomial_theta = binomial_tree_theta(S0, K, T, r, sigma, N, option_type)
    binomial_rho = binomial_tree_rho(S0, K, T, r, sigma, N, option_type)

    # Crank-Nicholson Model
    crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_delta = crank_nicholson_delta(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_gamma = crank_nicholson_gamma(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_vega = crank_nicholson_vega(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_theta = crank_nicholson_theta(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_rho = crank_nicholson_rho(S0, K, T, r, sigma, S_max, M, N, option_type)

    # Least Squares Monte Carlo Model
    lsm_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, N, option_type)
    lsm_delta = least_squares_monte_carlo_delta(S0, K, r, sigma, T, M, N, option_type)
    lsm_gamma = least_squares_monte_carlo_gamma(S0, K, r, sigma, T, M, N, option_type)
    lsm_vega = least_squares_monte_carlo_vega(S0, K, r, sigma, T, M, N, option_type)
    lsm_theta = least_squares_monte_carlo_theta(S0, K, r, sigma, T, M, N, option_type)
    lsm_rho = least_squares_monte_carlo_rho(S0, K, r, sigma, T, M, N, option_type)

    # Create and print a comparative table
    print(f"Option Pricing Comparison ({option_type.capitalize()} Option)")
    print(f"{'Model':<25}{'Price':<15}{'Delta':<15}{'Gamma':<15}{'Vega':<15}{'Theta':<15}{'Rho':<15}")

    models = ['Binomial Tree', 'Crank-Nicholson', 'Least Squares Monte Carlo']
    prices = [binomial_price, crank_price, lsm_price]
    deltas = [binomial_delta, crank_delta, lsm_delta]
    gammas = [binomial_gamma, crank_gamma, lsm_gamma]
    vegas = [binomial_vega, crank_vega, lsm_vega]
    thetas = [binomial_theta, crank_theta, lsm_theta]
    rhos = [binomial_rho, crank_rho, lsm_rho]

    for i in range(3):
        print(f"{models[i]:<25}{prices[i]:<15.4f}{deltas[i]:<15.4f}{gammas[i]:<15.4f}{vegas[i]:<15.4f}{thetas[i]:<15.4f}{rhos[i]:<15.4f}")

# Example input parameters
S0 = 590.5 # Current stock price
K = 600   # Strike price
T = 30/365    # Time to maturity in years
r = 0.0423  # Risk-free rate
sigma = 0.12 # Volatility
N = 100  # Number of periods (for Binomial and Crank-Nicholson)
M = 100  # Number of steps for Least Squares Monte Carlo
S_max = 1000  # Max stock price for Crank-Nicholson

# Run the comparative analysis
compare_option_pricing(S0, K, T, r, sigma, N, M, S_max, option_type="call")


Option Pricing Comparison (Call Option)
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            4.9889         0.3660         0.0177         0.9626         -54.7298       18.6798        
Crank-Nicholson          4.8131         0.3604         0.0289         -0.1146        535589712232517833946858727828246160646087077774293712794024960057399261753397125033571002575925661513130576656528232644150485417918969194835743644592288795177594152804650840902079736776300145569065551142289670156116102742016.000018.4396        
Least Squares Monte Carlo7.1193         0.4047         0.0143         20.1096        nan            11.9482        


<ipython-input-47-5d072cd24c11>:7: RuntimeWarning: invalid value encountered in sqrt
  diffusion = sigma * np.sqrt(dt) * np.random.normal(size=(M, N))


In [ ]:
import math
import numpy as np

# Placeholder functions for the Greeks and prices (to be replaced with actual implementations)

# Function to compare Greeks for each model
def compare_option_pricing(S0, K, T, r, sigma, N, M, S_max, option_type="call"):
    # Binomial Tree Model
    binomial_price = binomial_tree_american(S0, K, T, r, sigma, N, option_type)
    binomial_delta = binomial_tree_delta(S0, K, T, r, sigma, N, option_type)
    binomial_gamma = binomial_tree_gamma(S0, K, T, r, sigma, N, option_type)
    binomial_vega = binomial_tree_vega(S0, K, T, r, sigma, N, option_type)
    binomial_theta = binomial_tree_theta(S0, K, T, r, sigma, N, option_type)
    binomial_rho = binomial_tree_rho(S0, K, T, r, sigma, N, option_type)

    # Crank-Nicholson Model
    crank_price = crank_nicholson_american(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_delta = crank_nicholson_delta(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_gamma = crank_nicholson_gamma(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_vega = crank_nicholson_vega(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_theta = crank_nicholson_theta(S0, K, T, r, sigma, S_max, M, N, option_type)
    crank_rho = crank_nicholson_rho(S0, K, T, r, sigma, S_max, M, N, option_type)

    # Least Squares Monte Carlo Model
    lsm_price = least_squares_monte_carlo(S0, K, r, sigma, T, M, N, option_type)
    lsm_delta = least_squares_monte_carlo_delta(S0, K, r, sigma, T, M, N, option_type)
    lsm_gamma = least_squares_monte_carlo_gamma(S0, K, r, sigma, T, M, N, option_type)
    lsm_vega = least_squares_monte_carlo_vega(S0, K, r, sigma, T, M, N, option_type)
    lsm_theta = least_squares_monte_carlo_theta(S0, K, r, sigma, T, M, N, option_type)
    lsm_rho = least_squares_monte_carlo_rho(S0, K, r, sigma, T, M, N, option_type)

    # Print results
    print(f"K={K}")
    print(f"{'Model':<25}{'Price':<15}{'Delta':<15}{'Gamma':<15}{'Vega':<15}{'Theta':<15}{'Rho':<15}")
    models = ['Binomial Tree', 'Crank-Nicholson', 'Least Squares Monte Carlo']
    prices = [binomial_price, crank_price, lsm_price]
    deltas = [binomial_delta, crank_delta, lsm_delta]
    gammas = [binomial_gamma, crank_gamma, lsm_gamma]
    vegas = [binomial_vega, crank_vega, lsm_vega]
    thetas = [binomial_theta, crank_theta, lsm_theta]
    rhos = [binomial_rho, crank_rho, lsm_rho]

    for i in range(3):
        print(f"{models[i]:<25}{prices[i]:<15.4f}{deltas[i]:<15.4f}{gammas[i]:<15.4f}{vegas[i]:<15.4f}{thetas[i]:<15.4f}{rhos[i]:<15.4f}")

# Loop over the range of strike prices
S0 = 590.5  # Spot price
T = 21 / 365  # Time to expiry in years
r = 0.0423  # Risk-free rate
sigma = 0.14 # Volatility
N = 1000  # Steps for Binomial
M = 100  # Steps for Monte Carlo
S_max = 1200  # Maximum stock price for Crank-Nicholson
a=[607.81,604.68,602.8,607.46,604.33,604.21]
for K in range(a):  # Strike prices from 588 to 612
    T=T-(1/365)
    compare_option_pricing(K, 600, T, r, sigma, N, M, S_max, option_type="call")


<ipython-input-47-5d072cd24c11>:7: RuntimeWarning: invalid value encountered in sqrt
  diffusion = sigma * np.sqrt(dt) * np.random.normal(size=(M, N))


K=588
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            11.8640        0.5831         0.0164         1.1145         -70.5064       19.2506        
Crank-Nicholson          11.7441        0.5849         0.0205         0.1688         nan            19.2514        
Least Squares Monte Carlo12.4601        0.5937         0.0235         -0.0374        nan            11.9610        
K=589
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            11.3019        0.5667         0.0166         1.1154         -70.4773       19.2834        
Crank-Nicholson          11.2814        0.5680         0.0203         0.1686         nan            19.2841        
Least Squares Monte Carlo11.9890        0.5767         0.0150         0.0976         nan            11.9814        
K=590
Model                    Price          Delta         

In [ ]:
crank_delta = crank_nicholson_delta(607.81, 600, 14/365, 0.0423, sigma, S_max, M, N, 'call')
print(crank_delta)
#0.7151836009007108
T = 11 / 365  # Time to expiry in years
r = 0.0423  # Risk-free rate
sigma = 0.14 # Volatility
N = 1000  # Steps for Binomial
M = 100  # Steps for Monte Carlo
S_max = 1200  # Maximum stock price for Crank-Nicholson
a=[604.68,602.8,607.46,604.33,604.21]
op=[]
for K in a:  # Strike prices from 588 to 612
    T=T-(1/365)
    c=crank_nicholson_delta(K, 600, T, r, sigma, N, M, S_max, option_type="call")
    op=op+[crank_nicholson_american(K, 600, T, r, sigma, N, M, S_max, option_type="call")]
    print(c)
print(op)

0.7151836009007108
0.656697530158951
0.600005224889889
0.7582791383391057
0.664626734112155
0.6699972100811874
[8.663118450612336, 7.061971419248022, 9.979097384581333, 7.45126491650445, 7.0061686076230565]


In [ ]:
for K in range(588, 613):  # Strike prices from 588 to 612
    compare_option_pricing(S0, K, T, r, sigma, N, M, S_max, option_type="put")

<ipython-input-47-5d072cd24c11>:7: RuntimeWarning: invalid value encountered in sqrt
  diffusion = sigma * np.sqrt(dt) * np.random.normal(size=(M, N))


K=588
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            2.9397         -0.3970        0.0357         0.6648         -121.1449      -4.8232        
Crank-Nicholson          2.6257         -0.3757        0.0525         0.5512         nan            -4.8951        
Least Squares Monte Carlo2.8126         -0.4069        0.0385         -0.3773        nan            -5.2169        


<ipython-input-48-ae77f97f850d>:58: RuntimeWarning: invalid value encountered in matmul
  rhs = B @ option_values[1:-1]


K=589
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            3.3559         -0.4326        0.0365         0.6654         -123.8662      -4.9124        
Crank-Nicholson          3.2717         -0.4119        0.0498         0.5512         nan            -4.9272        
Least Squares Monte Carlo3.2028         -0.4341        0.0348         -0.2255        nan            -5.3247        
K=590
Model                    Price          Delta          Gamma          Vega           Theta          Rho            
Binomial Tree            3.8099         -0.4684        0.0370         0.6659         -125.1866      -4.9942        
Crank-Nicholson          3.9179         -0.4481        0.0472         0.5512         nan            -4.9593        
Least Squares Monte Carlo3.6949         -0.4666        0.0342         -0.1236        nan            -5.4269        
K=591
Model                    Price          Delta         

KeyboardInterrupt: 

In [ ]:
# Inputs
strike_price = 600
contract_size = 100
spot_prices = [604.68, 602.8, 607.46, 604.33, 604.21]  # From 9th Dec to 13th Dec
prev_spot_price = 607.81  # Price on 6th Dec
deltas = [0.7151836, 0.6566975, 0.6000052, 0.7582791, 0.6646267, 0.6699972]  # Including 6th Dec
option_prices = [8.663118450612336, 7.061971419248022, 9.979097384581333, 7.45126491650445, 7.0061686076230565]
option_position = 1  # Number of call options held

# Initialize variables
hedge_shares = deltas[0] * contract_size  # Initial shares sold on 6th Dec
cash_balance = -hedge_shares * prev_spot_price  # Proceeds from selling SPY shares
portfolio_values = []

# Loop through each day to calculate portfolio value
for i, spot_price in enumerate(spot_prices):
    # Delta for the current day
    current_delta = deltas[i + 1]

    # Hedge adjustment (buy/sell shares)
    required_shares = current_delta * contract_size
    shares_to_adjust = required_shares - hedge_shares
    cash_balance -= shares_to_adjust * spot_price  # Update cash balance

    # Update hedge shares
    hedge_shares = required_shares

    # Calculate option value using provided option prices
    option_value = option_prices[i] * contract_size * option_position

    # Calculate hedge value
    hedge_value = -hedge_shares * spot_price

    # Net portfolio value
    net_portfolio_value = option_value + hedge_value #cash_balance
    portfolio_values.append(net_portfolio_value)

# Output results
print("Net portfolio values:", portfolio_values)


Net portfolio values: [-38842.87258493877, -35462.116314075196, -45064.51247014187, -39420.25886944956, -39781.28396043769]


In [ ]:
# Inputs
strike_price = 600
contract_size = 100
spot_prices = [604.68, 602.8, 607.46, 604.33, 604.21]  # From 9th Dec to 13th Dec
prev_spot_price = 607.81  # Price on 6th Dec
deltas = [0.7151836, 0.6566975, 0.6000052, 0.7582791, 0.6646267, 0.6699972]  # Including 6th Dec
option_prices = [8.663118450612336, 7.061971419248022, 9.979097384581333, 7.45126491650445, 7.0061686076230565]
option_position = 1  # Number of call options held

# Initialize variables
hedge_shares = deltas[0] * contract_size  # Initial shares sold on 6th Dec
cash_balance = -hedge_shares * prev_spot_price  # Proceeds from selling SPY shares
hedging_cost = 0  # Total cost of hedging
portfolio_values = []

# Loop through each day to calculate portfolio value and hedging cost
for i, spot_price in enumerate(spot_prices):
    # Delta for the current day
    current_delta = deltas[i + 1]

    # Hedge adjustment (buy/sell shares)
    required_shares = current_delta * contract_size
    shares_to_adjust = required_shares - hedge_shares
    daily_cash_flow = shares_to_adjust * spot_price
    cash_balance -= daily_cash_flow  # Update cash balance
    hedging_cost += abs(daily_cash_flow)  # Accumulate absolute cost of adjustments

    # Update hedge shares
    hedge_shares = required_shares

    # Calculate option value using provided option prices
    option_value = option_prices[i] * contract_size * option_position

    # Calculate hedge value
    hedge_value = -hedge_shares * spot_price

    # Net portfolio value
    net_portfolio_value = option_value + hedge_value + cash_balance
    portfolio_values.append(net_portfolio_value)

# Output results
initial_portfolio_value = portfolio_values[0]
final_portfolio_value = portfolio_values[-1]
portfolio_change = final_portfolio_value - initial_portfolio_value

print("Net portfolio values:", portfolio_values)
print("Change in portfolio value:", portfolio_change)
print("Total hedging cost:", hedging_cost)


Net portfolio values: [-78775.90948173878, -71977.7413668752, -91194.64385234186, -79890.69476244957, -80576.21083393769]
Change in portfolio value: -1800.301352198905
Total hedging cost: 22552.642137899995


In [ ]:
import pandas as pd

# Inputs
strike_price = 600
contract_size = 100
spot_prices = [604.68, 602.8, 607.46, 604.33, 604.21]  # From 9th Dec to 13th Dec
prev_spot_price = 607.81  # Price on 6th Dec
deltas = [0.7151836, 0.6566975, 0.6000052, 0.7582791, 0.6646267, 0.6699972]  # Including 6th Dec
option_prices = [6.61000061035156,6.61000061035156, 9.01000022888184, 6.39000034332275, 5.59000015258789]
option_position = 1  # Number of call options held
dates = ["9 Dec", "10 Dec", "11 Dec", "12 Dec", "13 Dec"]

# Initialize variables
hedge_shares = deltas[0] * contract_size  # Initial shares sold on 6th Dec
cash_balance = -hedge_shares * prev_spot_price  # Proceeds from selling SPY shares
hedging_cost = 0  # Total cost of hedging
portfolio_values = []

# Data for the table
table_data = []

# Loop through each day to calculate portfolio value and hedging cost
for i, spot_price in enumerate(spot_prices):
    # Delta for the current day
    current_delta = deltas[i + 1]

    # Hedge adjustment (buy/sell shares)
    required_shares = current_delta * contract_size
    shares_to_adjust = required_shares - hedge_shares
    daily_cash_flow = shares_to_adjust * spot_price
    cash_balance -= daily_cash_flow  # Update cash balance
    hedging_cost += abs(daily_cash_flow)  # Accumulate absolute cost of adjustments

    # Update hedge shares
    hedge_shares = required_shares

    # Calculate option value using provided option prices
    option_value = option_prices[i] * contract_size * option_position

    # Calculate hedge value
    hedge_value = -hedge_shares * spot_price

    # Net portfolio value
    net_portfolio_value = option_value + hedge_value + cash_balance
    portfolio_values.append(net_portfolio_value)

    # Append data for the table
    table_data.append({
        "Date": dates[i],
        "Spot Price": spot_price,
        "Delta": current_delta,
        "Shares Adjusted": shares_to_adjust,
        "Cash Flow": daily_cash_flow,
        "Cumulative Hedging Cost": hedging_cost,
        "Option Value": option_value,
        "Hedge Value": hedge_value,
        "Net Portfolio Value": net_portfolio_value
    })

# Create a DataFrame for the table
df = pd.DataFrame(table_data)

# Display the table
print(df)


     Date  Spot Price     Delta  Shares Adjusted    Cash Flow  \
0   9 Dec      604.68  0.656698         -5.84861 -3536.537495   
1  10 Dec      602.80  0.600005         -5.66923 -3417.411844   
2  11 Dec      607.46  0.758279         15.82739  9614.506329   
3  12 Dec      604.33  0.664627         -9.36524 -5659.695489   
4  13 Dec      604.21  0.669997          0.53705   324.490980   

   Cumulative Hedging Cost  Option Value   Hedge Value  Net Portfolio Value  
0              3536.537495    661.000061 -39709.184430        -78981.221266  
1              6953.949339    661.000061 -36168.313456        -72022.938448  
2             16568.455668    901.000023 -46062.422209        -91291.553568  
3             22228.151157    639.000034 -40165.385361        -79996.821220  
4             22552.642138    559.000015 -40481.900821        -80717.827679  


In [ ]:
import pandas as pd

# Inputs
spot_prices = [604.68, 602.8, 607.46, 604.33, 604.21]  # From 9th Dec to 13th Dec
prev_spot_price = 607.81  # Price on 6th Dec
option_prices = [8.663118450612336, 7.061971419248022, 9.979097384581333, 7.45126491650445, 7.0061686076230565]
deltas = [0.7151836, 0.6566975, 0.6000052, 0.7582791, 0.6646267, 0.6699972]  # Including 6th Dec
contract_size = 100
dates = ["9 Dec", "10 Dec", "11 Dec", "12 Dec", "13 Dec"]

# Initialize variables
hedging_costs = []

# Calculate cost of hedging for each day
for i in range(len(spot_prices)):
    # Spot price change
    delta_spot = spot_prices[i] - (prev_spot_price if i == 0 else spot_prices[i - 1])

    # Option price change
    delta_option_price = option_prices[i] - (option_prices[i - 1] if i > 0 else 0)

    # Delta of the day
    delta = deltas[i + 1]  # Use i + 1 because deltas include the previous day's delta

    # Calculate cost of hedging
    hedging_cost = (delta * delta_spot * contract_size) - (delta_option_price * contract_size)
    hedging_costs.append(abs(hedging_cost))

# Create a DataFrame for better readability
table_data = {
    "Date": dates,
    "Spot Price": spot_prices,
    "Option Price": option_prices,
    "Delta": deltas[1:],  # Skip initial delta (6th Dec)
    "Spot Price Change": [spot_prices[i] - (prev_spot_price if i == 0 else spot_prices[i - 1]) for i in range(len(spot_prices))],
    "Option Price Change": [option_prices[i] - (option_prices[i - 1] if i > 0 else 0) for i in range(len(option_prices))],
    "Hedging Cost": hedging_costs,
}

df = pd.DataFrame(table_data)

# Output the table
print(df)

# Total hedging cost
total_hedging_cost = sum(hedging_costs)
print(f"\nTotal Hedging Cost: {total_hedging_cost}")


     Date  Spot Price  Option Price     Delta  Spot Price Change  \
0   9 Dec      604.68      8.663118  0.656698              -3.13   
1  10 Dec      602.80      7.061971  0.600005              -1.88   
2  11 Dec      607.46      9.979097  0.758279               4.66   
3  12 Dec      604.33      7.451265  0.664627              -3.13   
4  13 Dec      604.21      7.006169  0.669997              -0.12   

   Option Price Change  Hedging Cost  
0             8.663118   1071.858163  
1            -1.601147     47.313726  
2             2.917126     61.645464  
3            -2.527832     44.755090  
4            -0.445096     36.469664  

Total Hedging Cost: 1262.0421063601673
